In [23]:
import pandas as pd
from pulp import LpProblem, LpMaximize, LpVariable, lpSum

In [24]:
class Bess_Optimizer:
    def __init__(self, power_capacity = 100, energy_capacity = 200, efficiency = 0.9):
        self.power_capacity = power_capacity
        self.energy_capacity = energy_capacity
        self.energy_prices = None
        self.reg_up_prices = None
        self.reg_down_prices = None
        self.efficiency = efficiency

    def load_prices(self, energy_price_file):
        '''
        load energy prices from energy_price_file
        '''
        
        self.energy_prices = pd.read_csv(energy_price_file)
        self.energy_prices['date'] = pd.to_datetime(self.energy_prices['date'])

        
    def load_regulation(self):
        pass

    def optimize_period(self, start_day = None, end_day = None, initial_charge = 0):
        self.optimizer = LpProblem('Bess-Fluence', LpMaximize)                                              #set the model
        
        start_day = pd.to_datetime(start_day)
        end_day = pd.to_datetime(end_day)
        period = self.energy_prices[(self.energy_prices['date'] >= start_day) & (self.energy_prices['date'] <= end_day)]['date']
        period_str = [str(p) for p in period]

        print(period_str)
        
        #Decision Variables
        self.energy_hourly = LpVariable.dicts(                                                                 
                                    name = 'energy', 
                                    indices= period_str,
                                    lowBound = -self.power_capacity, 
                                    upBound = self.power_capacity
                                    )      
        
        self.reg_up_power = LpVariable.dicts(
                                        name = 'reg_up_power',
                                        indices= period_str,
                                        lowBound = 0,
                                        upBound = self.power_capacity
                                        )
                                
        self.reg_down_power = LpVariable.dicts(
                                        name = 'reg_down_power',
                                        indices= period_str,
                                        lowBound = 0,
                                        upBound = self.power_capacity
                                        )

        self.energy_charge = LpVariable.dicts(
                                        name = 'charge',
                                        indices= period_str,
                                        lowBound = 0,
                                        upBound = self.energy_capacity
                                        )
        

        self.energy_prices.index.isin(period)
        
        
        # Objective functions
        self.optimizer += lpSum(
                                self.energy_prices[self.energy_prices['date']==[pd.to_datetime(hour)]].values[0] * self.energy_hourly[hour]
                                for hour in period_str
                                )
        
        solution = self.optimizer.solve()
        print(solution)

In [25]:
optimizer = Bess_Optimizer()
optimizer.load_prices( energy_price_file ='data/energy_price.csv')
optimizer.optimize_period(start_day = '1/1/2025', end_day = '1/2/2025')


['2025-01-01 00:00:00', '2025-01-01 01:00:00', '2025-01-01 02:00:00', '2025-01-01 03:00:00', '2025-01-01 04:00:00', '2025-01-01 05:00:00', '2025-01-01 06:00:00', '2025-01-01 07:00:00', '2025-01-01 08:00:00', '2025-01-01 09:00:00', '2025-01-01 10:00:00', '2025-01-01 11:00:00', '2025-01-01 12:00:00', '2025-01-01 13:00:00', '2025-01-01 14:00:00', '2025-01-01 15:00:00', '2025-01-01 16:00:00', '2025-01-01 17:00:00', '2025-01-01 18:00:00', '2025-01-01 19:00:00', '2025-01-01 20:00:00', '2025-01-01 21:00:00', '2025-01-01 22:00:00', '2025-01-01 23:00:00', '2025-01-02 00:00:00']


ValueError: ('Lengths must match to compare', (168,), (1,))